# Phase 2.2 Experiment 4: Compare three GTZAN dataset filters

## Research question

Compare the Phase 2.2 CNN trained and evaluated on (1) the full eligible dataset, (2) the tracks in the partially filtered manifest, and (3) the tracks in the fully filtered manifest, which also removes reported mislabeled tracks.

## Matched experiment settings

All three runs use the same five-block CNN, 30-second tracks converted to six 5-second log-mel windows, cross-entropy loss, AdamW optimizer, learning-rate scheduler, early stopping, track-level probability averaging, random seed, and nominal track split of 81% train, 9% validation, and 10% test.

The full-data control uses every eligible track. The other two runs use only paths in their respective manifest for all three splits. The manifests are loaded from this repository in Colab or from the local project checkout.

Each condition receives its own stratified, hash-grouped split. As a result, each model is evaluated only on its condition's test tracks, and the held-out tracks differ across conditions. Compare the metrics as signals for this experiment, alongside the class-level tables; they are not paired predictions on identical songs. Stratified grouping and duplicate groups can make the realized percentages differ slightly from the nominal percentages.

Run the cells from top to bottom. The first run downloads the GTZAN archive (about 1.23 GB) if audio is not already present and caches reusable features under artifacts.

## 1. Download and prepare GTZAN

This data setup follows the Phase 2.2 baseline: it downloads the Marsyas GTZAN archive when needed, extracts the ten genre folders, and skips the known corrupted jazz.00054.wav file.

In [ ]:
import importlib.util
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path, PurePosixPath

if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
from huggingface_hub import hf_hub_download

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_DIR = Path("/content")
else:
    current_and_parent_dirs = [Path.cwd(), *Path.cwd().parents]
    PROJECT_DIR = next(
        (path for path in current_and_parent_dirs if (path / "filtered_splits").is_dir()),
        Path.cwd(),
    )
DATA_DIR = PROJECT_DIR / "data" / "genres_original"
GENRE_FOLDERS = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock",
]

# returns a dictionary of each genre and how many files are stored under its name.
def genre_file_counts(root):
    return {
        genre: len(list((root / genre).glob("*.wav")))
        for genre in GENRE_FOLDERS
    }

# Creates the directory
# parents=True means create the directory if it doesn't exist
# exist_ok=True means reuse the directory if it exists already (don't raise an error)
DATA_DIR.mkdir(parents=True, exist_ok=True)
existing_counts = genre_file_counts(DATA_DIR)

# checks whether all values (counts) inside the genre counts dict are bigger or equal to 90.
# If so, skips the download
if all(count >= 90 for count in existing_counts.values()):
    print(f"GTZAN is already present at {DATA_DIR.resolve()} — skipping download.")
else:
    archive_path = hf_hub_download(
        repo_id="marsyas/gtzan",
        repo_type="dataset",
        filename="data/genres.tar.gz",
        local_dir=str(PROJECT_DIR / "data" / "downloads"),
    )
    print(f"Downloaded archive: {archive_path}")

    extracted_files = 0
    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in archive.getmembers():
            if not member.isfile():
                continue
            member_path = PurePosixPath(member.name)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in GTZAN archive: {member.name}")
            filename = member_path.name
            if not filename.lower().endswith(".wav") or filename.startswith("._"):
                continue
            genres_in_path = [
                part.lower() for part in member_path.parts[:-1]
                if part.lower() in GENRE_FOLDERS
            ]
            if not genres_in_path:
                continue
            genre = genres_in_path[-1]
            if genre == "jazz" and filename == "jazz.00054.wav":
                continue
            destination = DATA_DIR / genre / filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.extractfile(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
            extracted_files += 1

    final_counts = genre_file_counts(DATA_DIR)
    missing_or_incomplete = {genre: count for genre, count in final_counts.items() if count < 90}
    if missing_or_incomplete:
        raise RuntimeError(
            f"GTZAN extraction looks incomplete: {missing_or_incomplete}. "
            "Check the Colab internet connection and rerun this cell."
        )
    print(f"Extracted {extracted_files} usable WAV files into {DATA_DIR.resolve()}")
    print("Files per genre:", final_counts)

print(f"Dataset directory ready: {DATA_DIR.resolve()}")

## 2. Imports, settings, and reproducibility

In [ ]:
import hashlib
import math
import random
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from torch.utils.data import DataLoader, Dataset
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Deterministic kernels can make repeated runs easier to compare.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Audio and feature settings
PROJECT_DIR = PROJECT_DIR  # detected in the dataset setup cell above
DATA_DIR = PROJECT_DIR / "data" / "genres_original"
SAMPLE_RATE = 22_050
TRACK_SECONDS = 30
SEGMENT_SECONDS = 5
N_MELS = 128
N_FFT = 2_048
HOP_LENGTH = 512
DB_RANGE = 80.0

# Training settings
BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 8
LEARNING_RATE = 1e-3

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
CHECKPOINT_DIR = ARTIFACTS_DIR / "phase_2_2_experiment_4"

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Dataset folder: {DATA_DIR.resolve()}")

## 3. Index the audio and identify exact copies

In [ ]:
# A few common extraction layouts are checked before asking you to edit DATA_DIR.
if not DATA_DIR.exists():
    alternatives = [
        Path("genres_original"),
        Path("Data/genres_original"),
        Path("data/Data/genres_original"),
    ]
    DATA_DIR = next((path for path in alternatives if path.exists()), DATA_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Could not find the GTZAN genre folders at {DATA_DIR.resolve()}. "
        "Extract the dataset and set DATA_DIR to the folder that directly contains "
        "the ten genre folders."
    )

audio_extensions = {".wav", ".au", ".mp3", ".flac", ".ogg"}
genre_dirs = sorted(path for path in DATA_DIR.iterdir() if path.is_dir())
CLASS_NAMES = [path.name for path in genre_dirs]
if len(CLASS_NAMES) != 10:
    raise ValueError(
        f"Expected 10 genre folders, found {len(CLASS_NAMES)}: {CLASS_NAMES}. "
        "Check that DATA_DIR points to the folder containing the genre folders."
    )

class_to_id = {name: index for index, name in enumerate(CLASS_NAMES)}

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as audio_file:
        for block in iter(lambda: audio_file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

records = []
for genre_dir in genre_dirs:
    for audio_path in sorted(genre_dir.iterdir()):
        if audio_path.is_file() and audio_path.suffix.lower() in audio_extensions:
            try:
                audio_info = sf.info(str(audio_path))
                file_hash = sha256_file(audio_path)
            except (OSError, RuntimeError, ValueError) as error:
                print(f"Skipping unreadable audio file {audio_path}: {error}")
                continue
            records.append(
                {
                    "path": audio_path.resolve(),
                    "track_name": audio_path.name,
                    "relative_path": PurePosixPath(genre_dir.name, audio_path.name).as_posix(),
                    "genre": genre_dir.name,
                    "label_id": class_to_id[genre_dir.name],
                    "duration_seconds": audio_info.duration,
                    "sha256": file_hash,
                }
            )

tracks = pd.DataFrame(records)
if tracks.empty:
    raise ValueError(
        f"No supported audio files found directly inside the genre folders in {DATA_DIR}."
    )

print(f"Found {len(tracks)} audio files across {tracks['genre'].nunique()} genres.")
display(tracks.head())

## 4. Inspect class balance, durations, and conflicting exact copies

In [ ]:
print("Files per genre:")
display(tracks.groupby("genre").size().rename("file_count").to_frame())

print(
    "\nDuration summary (seconds): "
    f"min={tracks['duration_seconds'].min():.2f}, "
    f"median={tracks['duration_seconds'].median():.2f}, "
    f"max={tracks['duration_seconds'].max():.2f}"
)

all_indexed_manifest_paths = {
    str(path).replace("\\", "/").removeprefix("./").casefold()
    for path in tracks["relative_path"]
}

duplicate_mask = tracks.duplicated("sha256", keep=False)
duplicate_files = tracks.loc[duplicate_mask].sort_values(["sha256", "genre"])
conflicting_hashes = (
    tracks.groupby("sha256")["genre"].nunique().loc[lambda counts: counts > 1].index
)
print(f"\nFiles that are exact copies of another file: {int(duplicate_mask.sum())}")
print(f"Exact-content groups assigned conflicting genre labels: {len(conflicting_hashes)}")

if len(conflicting_hashes):
    print("Conflicting exact copies (excluded from this run):")
    display(duplicate_files[duplicate_files["sha256"].isin(conflicting_hashes)][
        ["track_name", "genre", "sha256"]
    ])

# A copy with conflicting labels has no unambiguous supervised target.
tracks = tracks.loc[~tracks["sha256"].isin(conflicting_hashes)].reset_index(drop=True)

print(f"\nFiles kept for modeling: {len(tracks)}")
display(tracks.groupby("genre").size().rename("usable_file_count").to_frame())

## 5. Load the partially and fully filtered track lists

The two text files contain the allowed relative paths for their respective filtered runs. In Colab, the notebook downloads these files from the repository. Locally, it reads the checked-out copies. The full list is expected to be a subset of the partial list; paths absent from the downloaded or indexed audio are reported and omitted.

In [ ]:
from urllib.request import urlopen

MANIFEST_FILENAMES = {
    "trained_on_partially_filtered_dataset": "partially_filtered_dataset.txt",
    "trained_on_fully_filtered_dataset": "fully_filtered_dataset.txt",
}
MANIFEST_LOCAL_DIR = PROJECT_DIR / "phase_2" / "filter_experiments"
RAW_MANIFEST_BASE = (
    "https://raw.githubusercontent.com/Rani1281/MusicGenreClassification/"
    "main/phase_2/filter_experiments/"
)

def normalize_manifest_path(path):
    return str(path).strip().replace("\\", "/").removeprefix("./").casefold()

manifest_paths = {}
manifest_sources = {}
manifest_line_counts = {}

for run_name, filename in MANIFEST_FILENAMES.items():
    local_path = MANIFEST_LOCAL_DIR / filename
    if not IN_COLAB and local_path.is_file():
        manifest_text = local_path.read_text(encoding="utf-8-sig")
        manifest_sources[run_name] = str(local_path.resolve())
    else:
        manifest_url = RAW_MANIFEST_BASE + filename
        with urlopen(manifest_url, timeout=60) as response:
            manifest_text = response.read().decode("utf-8-sig")
        manifest_sources[run_name] = manifest_url

    raw_paths = [
        normalize_manifest_path(line)
        for line in manifest_text.splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]
    manifest_line_counts[run_name] = len(raw_paths)
    manifest_paths[run_name] = set(raw_paths)

    if not manifest_paths[run_name]:
        raise ValueError(f"The manifest is empty: {filename}")
    if len(manifest_paths[run_name]) != len(raw_paths):
        raise ValueError(f"The manifest contains duplicate paths: {filename}")

partial_paths = manifest_paths["trained_on_partially_filtered_dataset"]
fully_paths = manifest_paths["trained_on_fully_filtered_dataset"]
if not fully_paths.issubset(partial_paths):
    outside_partial = sorted(fully_paths - partial_paths)
    raise ValueError(
        "The fully filtered manifest is expected to be a subset of the partial "
        f"manifest; found {len(outside_partial)} paths outside it."
    )

indexed_paths_before_conflict_exclusion = set(all_indexed_manifest_paths)
indexed_paths_after_conflict_exclusion = {
    normalize_manifest_path(path) for path in tracks["relative_path"]
}

manifest_summary_rows = []
for run_name, paths in manifest_paths.items():
    missing_from_audio_index = paths - indexed_paths_before_conflict_exclusion
    excluded_as_conflicting_label_copies = (
        paths & indexed_paths_before_conflict_exclusion
    ) - indexed_paths_after_conflict_exclusion
    available_paths = paths & indexed_paths_after_conflict_exclusion
    manifest_summary_rows.append(
        {
            "dataset": run_name,
            "manifest_entries": len(paths),
            "available_audio_tracks": len(available_paths),
            "not_found_in_audio_index": len(missing_from_audio_index),
            "excluded_for_conflicting_exact_copy_labels": len(
                excluded_as_conflicting_label_copies
            ),
        }
    )
    print(f"{run_name} manifest source: {manifest_sources[run_name]}")
    if missing_from_audio_index:
        print(
            f"  Paths not found in the extracted audio "
            f"({len(missing_from_audio_index)}): "
            f"{sorted(missing_from_audio_index)[:10]}"
        )
    if excluded_as_conflicting_label_copies:
        print(
            "  Paths excluded because identical audio has conflicting folder labels "
            f"({len(excluded_as_conflicting_label_copies)})."
        )

display(pd.DataFrame(manifest_summary_rows).set_index("dataset"))
print(
    "Additional paths removed by the fully filtered list compared with the partial "
    f"list: {len(partial_paths - fully_paths)}"
)

tracks["normalized_relative_path"] = tracks["relative_path"].map(normalize_manifest_path)
tracks["in_partially_filtered_manifest"] = tracks["normalized_relative_path"].isin(
    partial_paths
)
tracks["in_fully_filtered_manifest"] = tracks["normalized_relative_path"].isin(
    fully_paths
)

DATASET_LABELS = {
    "trained_on_full_dataset": "Full dataset (control)",
    "trained_on_partially_filtered_dataset": "Partially filtered",
    "trained_on_fully_filtered_dataset": "Fully filtered",
}
DATASET_TRACKS = {
    "trained_on_full_dataset": tracks.reset_index(drop=True).copy(),
    "trained_on_partially_filtered_dataset": tracks.loc[
        tracks["in_partially_filtered_manifest"]
    ].reset_index(drop=True).copy(),
    "trained_on_fully_filtered_dataset": tracks.loc[
        tracks["in_fully_filtered_manifest"]
    ].reset_index(drop=True).copy(),
}

for run_name, candidate_tracks in DATASET_TRACKS.items():
    print(
        f"{DATASET_LABELS[run_name]}: {len(candidate_tracks)} eligible tracks "
        f"across {candidate_tracks['genre'].nunique()} genres"
    )
    if candidate_tracks.empty or candidate_tracks["genre"].nunique() != len(CLASS_NAMES):
        raise ValueError(f"{DATASET_LABELS[run_name]} is missing one or more genres.")


## 6. Create an independent 81/9/10 split for each condition

Each candidate dataset is split at the original-track level before 5-second windows are created. The earlier fold-based split only approximated class balance, so this cell now assigns whole SHA-256 groups separately within each genre. A subset-sum allocation chooses the feasible group combination closest to the target counts for test (10%) and validation (9%); the remaining tracks are used for training (about 81%). Exact copies stay together. The following table shows actual per-genre counts and percentages, including any unavoidable difference caused by duplicate groups.

In [ ]:
SPLIT_PERCENTAGES = {"train": 81, "validation": 9, "test": 10}

def choose_hash_groups_near_target(group_items, target_count, random_state):
    # Choose whole hash groups with a total track count nearest the target.
    shuffled_items = list(group_items)
    random.Random(random_state).shuffle(shuffled_items)

    # Dynamic programming over attainable track counts; each hash group is used once.
    attainable_counts = {0: ()}
    for group_hash, group_size in shuffled_items:
        previous_states = list(attainable_counts.items())
        for current_count, chosen_hashes in previous_states:
            next_count = current_count + int(group_size)
            if next_count not in attainable_counts:
                attainable_counts[next_count] = chosen_hashes + (group_hash,)

    best_count = min(
        attainable_counts,
        key=lambda count: (abs(count - target_count), count > target_count),
    )
    return set(attainable_counts[best_count]), best_count


def make_track_split(candidate_tracks, random_state=SEED):
    candidate_tracks = candidate_tracks.reset_index(drop=True).copy()
    class_counts = (
        candidate_tracks.groupby("genre").size().reindex(CLASS_NAMES, fill_value=0)
    )
    if (class_counts < 10).any():
        raise ValueError(
            "Each condition needs at least 10 tracks in every genre for the "
            f"81/9/10 split. Counts: {class_counts.to_dict()}"
        )

    split_by_hash = {}
    allocation_rows = []
    test_fraction = SPLIT_PERCENTAGES["test"] / 100
    validation_fraction = SPLIT_PERCENTAGES["validation"] / 100

    for genre_index, genre in enumerate(CLASS_NAMES):
        genre_tracks = candidate_tracks.loc[candidate_tracks["genre"] == genre]
        group_sizes = genre_tracks.groupby("sha256").size()
        group_items = list(group_sizes.items())
        genre_total = len(genre_tracks)
        test_target = int(np.floor(genre_total * test_fraction + 0.5))
        validation_target = int(np.floor(genre_total * validation_fraction + 0.5))

        test_hashes, test_count = choose_hash_groups_near_target(
            group_items,
            test_target,
            random_state + genre_index * 100 + 1,
        )
        for group_hash in test_hashes:
            split_by_hash[group_hash] = "test"

        remaining_groups = [
            (group_hash, group_size)
            for group_hash, group_size in group_items
            if group_hash not in test_hashes
        ]
        validation_hashes, validation_count = choose_hash_groups_near_target(
            remaining_groups,
            validation_target,
            random_state + genre_index * 100 + 2,
        )
        for group_hash in validation_hashes:
            split_by_hash[group_hash] = "validation"
        for group_hash, _ in remaining_groups:
            if group_hash not in validation_hashes:
                split_by_hash[group_hash] = "train"

        train_count = genre_total - test_count - validation_count
        allocation_rows.append(
            {
                "genre": genre,
                "total_tracks": genre_total,
                "train_tracks": train_count,
                "train_percent": 100 * train_count / genre_total,
                "validation_tracks": validation_count,
                "validation_percent": 100 * validation_count / genre_total,
                "test_tracks": test_count,
                "test_percent": 100 * test_count / genre_total,
                "target_test_tracks": test_target,
                "target_validation_tracks": validation_target,
            }
        )

    if len(split_by_hash) != candidate_tracks["sha256"].nunique():
        raise AssertionError("A hash group did not receive exactly one split.")

    split_assignment = candidate_tracks[["genre", "sha256"]].copy()
    split_assignment["split"] = split_assignment["sha256"].map(split_by_hash)
    if split_assignment["split"].isna().any():
        raise AssertionError("Some tracks were not assigned to a split.")

    split_count_per_hash = split_assignment.groupby("sha256")["split"].nunique()
    if split_count_per_hash.max() != 1:
        raise AssertionError("An exact-content group crossed a split.")

    split_tracks = {
        split_name: candidate_tracks.loc[
            split_assignment["split"] == split_name
        ].reset_index(drop=True)
        for split_name in ("train", "validation", "test")
    }
    return split_tracks, split_assignment, pd.DataFrame(allocation_rows)


DATASET_SPLITS = {}
split_summary_rows = []
genre_split_rows = []
for run_name, candidate_tracks in DATASET_TRACKS.items():
    splits, assignments, genre_allocations = make_track_split(candidate_tracks)
    DATASET_SPLITS[run_name] = {
        "tracks": candidate_tracks,
        "splits": splits,
        "assignments": assignments,
        "genre_allocations": genre_allocations,
    }

    total_tracks = len(candidate_tracks)
    summary = {"dataset": DATASET_LABELS[run_name], "total_tracks": total_tracks}
    for split_name in ("train", "validation", "test"):
        count = len(splits[split_name])
        summary[f"{split_name}_tracks"] = count
        summary[f"{split_name}_percent"] = 100 * count / total_tracks
    split_summary_rows.append(summary)

    for allocation in genre_allocations.to_dict(orient="records"):
        allocation["dataset"] = DATASET_LABELS[run_name]
        genre_split_rows.append(allocation)

split_summary = pd.DataFrame(split_summary_rows).set_index("dataset")
display(split_summary.round(1))
print(
    "Target split percentages:",
    ", ".join(f"{name} {value}%" for name, value in SPLIT_PERCENTAGES.items()),
)

genre_split_summary = pd.DataFrame(genre_split_rows).set_index(["dataset", "genre"])
print("Per-genre split counts and realized percentages")
display(genre_split_summary.round(1))


## 7. Convert tracks to cached log-mel spectrograms

In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SAMPLES = SAMPLE_RATE * TRACK_SECONDS
EXPECTED_FRAMES = 1 + EXPECTED_SAMPLES // HOP_LENGTH
NUM_SEGMENTS = math.ceil(TRACK_SECONDS / SEGMENT_SECONDS)
FRAMES_PER_SEGMENT = math.ceil(EXPECTED_FRAMES / NUM_SEGMENTS)

FEATURE_CACHE_DIR = (
    ARTIFACTS_DIR
    / "gtzan_log_mel"
    / f"sr{SAMPLE_RATE}_mels{N_MELS}_fft{N_FFT}_hop{HOP_LENGTH}_sec{TRACK_SECONDS}_db{int(DB_RANGE)}"
)
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def feature_path_for_hash(file_hash):
    return FEATURE_CACHE_DIR / file_hash[:2] / f"{file_hash}.npy"

def extract_log_mel(audio_path):
    audio, _ = librosa.load(
        str(audio_path),
        sr=SAMPLE_RATE,
        mono=True,
        duration=TRACK_SECONDS,
    )
    if len(audio) < EXPECTED_SAMPLES:
        audio = np.pad(audio, (0, EXPECTED_SAMPLES - len(audio)))
    else:
        audio = audio[:EXPECTED_SAMPLES]

    power_mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0,
    )
    decibels = librosa.power_to_db(power_mel, ref=np.max, top_db=DB_RANGE)
    scaled = np.clip((decibels + DB_RANGE) / DB_RANGE, 0.0, 1.0).astype(np.float32)

    # Keep a predictable number of frames even if an audio backend differs by one frame.
    if scaled.shape[1] < EXPECTED_FRAMES:
        scaled = np.pad(scaled, ((0, 0), (0, EXPECTED_FRAMES - scaled.shape[1])))
    else:
        scaled = scaled[:, :EXPECTED_FRAMES]
    return scaled

unique_tracks = tracks.drop_duplicates("sha256")
for position, row in enumerate(unique_tracks.itertuples(index=False), start=1):
    cache_path = feature_path_for_hash(row.sha256)
    if not cache_path.exists():
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(cache_path, extract_log_mel(row.path))
    if position % 100 == 0 or position == len(unique_tracks):
        print(f"Prepared features for {position}/{len(unique_tracks)} unique tracks")

example_mel = np.load(feature_path_for_hash(unique_tracks.iloc[0]["sha256"]))
print(f"Feature cache: {FEATURE_CACHE_DIR.resolve()}")
print(f"One track feature shape: {example_mel.shape} (mel bands × time frames)")

## 8. Build a separate set of loaders for each model

Every run receives its own train, validation, and test loaders from its split. Data preparation and windowing match the Phase 2.2 baseline; validation and test loader order is fixed, and training windows are shuffled with the same seed.

In [ ]:
class MelWindowDataset(Dataset):
    def __init__(self, track_table):
        self.track_rows = track_table.to_dict(orient="records")
        self.examples = [
            (track_index, segment_index)
            for track_index, row in enumerate(self.track_rows)
            for segment_index in range(NUM_SEGMENTS)
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, item_index):
        track_index, segment_index = self.examples[item_index]
        row = self.track_rows[track_index]
        mel = np.load(feature_path_for_hash(row["sha256"]), mmap_mode="r")

        start = segment_index * FRAMES_PER_SEGMENT
        window = np.array(
            mel[:, start : start + FRAMES_PER_SEGMENT],
            dtype=np.float32,
            copy=True,
        )
        if window.shape[1] < FRAMES_PER_SEGMENT:
            window = np.pad(
                window,
                ((0, 0), (0, FRAMES_PER_SEGMENT - window.shape[1])),
            )

        features = torch.from_numpy(window[None, :, :])
        label = torch.tensor(row["label_id"], dtype=torch.long)
        source_track = torch.tensor(track_index, dtype=torch.long)
        return features, label, source_track


def make_train_loader(dataset):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )


EXPERIMENT_RUNS = {}
for run_name, dataset_split in DATASET_SPLITS.items():
    train_dataset = MelWindowDataset(dataset_split["splits"]["train"])
    validation_dataset = MelWindowDataset(dataset_split["splits"]["validation"])
    test_dataset = MelWindowDataset(dataset_split["splits"]["test"])

    EXPERIMENT_RUNS[run_name] = {
        "train_dataset": train_dataset,
        "validation_dataset": validation_dataset,
        "test_dataset": test_dataset,
        "train_loader": make_train_loader(train_dataset),
        "validation_loader": DataLoader(
            validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
        ),
        "test_loader": DataLoader(
            test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
        ),
    }

    print(
        f"{DATASET_LABELS[run_name]} | "
        f"train windows: {len(train_dataset)} | "
        f"validation windows: {len(validation_dataset)} | "
        f"test windows: {len(test_dataset)}"
    )


## 9. Define the Phase 2.2 CNN

In [ ]:
class SmallAudioCNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()
        self.filters = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(p=0.30),
            nn.Linear(256, number_of_classes),
        )

    def forward(self, inputs):
        return self.classifier(self.filters(inputs))

## 10. Aggregate window probabilities to track predictions

In [ ]:
@torch.inference_mode()
def collect_predictions(model, data_loader, dataset):
    model.eval()
    all_probabilities = []
    all_window_labels = []
    all_track_indices = []

    for features, labels, track_indices in data_loader:
        logits = model(features.to(DEVICE))
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()
        all_probabilities.append(probabilities)
        all_window_labels.append(labels.numpy())
        all_track_indices.append(track_indices.numpy())

    window_probabilities = np.concatenate(all_probabilities)
    window_true = np.concatenate(all_window_labels)
    window_track_indices = np.concatenate(all_track_indices)

    track_probabilities = np.zeros(
        (len(dataset.track_rows), len(CLASS_NAMES)),
        dtype=np.float64,
    )
    window_counts = np.zeros(len(dataset.track_rows), dtype=np.int64)
    np.add.at(track_probabilities, window_track_indices, window_probabilities)
    np.add.at(window_counts, window_track_indices, 1)
    track_probabilities /= window_counts[:, None]

    track_true = np.asarray(
        [row["label_id"] for row in dataset.track_rows],
        dtype=np.int64,
    )
    track_predicted = track_probabilities.argmax(axis=1)

    return {
        "window_probabilities": window_probabilities,
        "window_true": window_true,
        "window_predicted": window_probabilities.argmax(axis=1),
        "window_track_indices": window_track_indices,
        "track_probabilities": track_probabilities,
        "track_true": track_true,
        "track_predicted": track_predicted,
    }

## 11. Train all three models with the same protocol

The model architecture, initialization seed, loss, optimizer, learning rate, scheduler, epoch limit, patience, and validation checkpoint rule match the Phase 2.2 workflow. Each model selects its checkpoint using its own validation split and is evaluated on its own held-out test split. Checkpoints are saved separately under artifacts/phase_2_2_experiment_4.

In [ ]:
def reset_random_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_model(run_name, train_loader, validation_loader, validation_dataset):
    """Train the reference CNN and select its checkpoint on this run’s validation tracks."""
    reset_random_seeds(SEED)
    model = SmallAudioCNN(number_of_classes=len(CLASS_NAMES)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    history = []
    best_score = (-1.0, -1.0)
    best_epoch = 0
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        correct_windows = 0
        seen_windows = 0

        for features, labels_batch, _ in train_loader:
            features = features.to(DEVICE)
            labels_batch = labels_batch.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = criterion(logits, labels_batch)
            loss.backward()
            optimizer.step()

            batch_size = labels_batch.size(0)
            total_loss += loss.item() * batch_size
            correct_windows += (logits.argmax(dim=1) == labels_batch).sum().item()
            seen_windows += batch_size

        validation_result = collect_predictions(
            model, validation_loader, validation_dataset
        )
        validation_accuracy = accuracy_score(
            validation_result["track_true"], validation_result["track_predicted"]
        )
        validation_macro_f1 = f1_score(
            validation_result["track_true"],
            validation_result["track_predicted"],
            average="macro",
            zero_division=0,
        )
        scheduler.step(validation_accuracy)

        epoch_record = {
            "epoch": epoch,
            "train_loss": total_loss / seen_windows,
            "train_window_accuracy": correct_windows / seen_windows,
            "validation_track_accuracy": validation_accuracy,
            "validation_track_macro_f1": validation_macro_f1,
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(epoch_record)

        score = (validation_accuracy, validation_macro_f1)
        if score > best_score:
            best_score = score
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
        else:
            epochs_without_improvement += 1

        print(
            f"{run_name} | Epoch {epoch:02d}/{EPOCHS} | "
            f"loss {epoch_record['train_loss']:.4f} | "
            f"train window acc {epoch_record['train_window_accuracy']:.3f} | "
            f"validation track acc {validation_accuracy:.3f} | "
            f"validation macro-F1 {validation_macro_f1:.3f} | "
            f"lr {epoch_record['learning_rate']:.2e}"
        )

        if epochs_without_improvement >= PATIENCE:
            print(f"{run_name}: early stopping after {PATIENCE} epochs without improvement.")
            break

    if best_state is None:
        raise RuntimeError(f"{run_name} training did not produce a checkpoint.")

    model.load_state_dict(best_state)
    model.eval()
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint_path = CHECKPOINT_DIR / f"{run_name.lower().replace(' ', '_')}.pt"
    torch.save(
        {
            "model_state_dict": best_state,
            "class_names": CLASS_NAMES,
            "best_epoch": best_epoch,
            "sample_rate": SAMPLE_RATE,
            "track_seconds": TRACK_SECONDS,
            "segment_seconds": SEGMENT_SECONDS,
            "n_mels": N_MELS,
            "n_fft": N_FFT,
            "hop_length": HOP_LENGTH,
            "seed": SEED,
        },
        checkpoint_path,
    )
    print(f"{run_name}: best validation checkpoint at epoch {best_epoch}; saved to {checkpoint_path}")
    return model, pd.DataFrame(history), best_epoch


In [ ]:
experiment_models = {}
experiment_histories = {}
experiment_best_epochs = {}

for run_name, run_data in EXPERIMENT_RUNS.items():
    print(f"\nStarting {DATASET_LABELS[run_name]} run ({run_name})")
    model, history, best_epoch = train_model(
        run_name,
        run_data["train_loader"],
        run_data["validation_loader"],
        run_data["validation_dataset"],
    )
    experiment_models[run_name] = model
    experiment_histories[run_name] = history
    experiment_best_epochs[run_name] = best_epoch
    print(f"{DATASET_LABELS[run_name]} best epoch: {best_epoch}\n")


## 12. Compare learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for run_name, history_table in experiment_histories.items():
    label = DATASET_LABELS[run_name]
    axes[0].plot(
        history_table["epoch"],
        history_table["train_loss"],
        marker="o",
        label=label,
    )
    axes[1].plot(
        history_table["epoch"],
        history_table["validation_track_accuracy"],
        marker="o",
        label=f"{label} accuracy",
    )
    axes[1].plot(
        history_table["epoch"],
        history_table["validation_track_macro_f1"],
        marker="s",
        linestyle="--",
        label=f"{label} macro-F1",
    )

axes[0].set_title("Training loss (5-second windows)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].set_title("Validation track metrics (condition-specific splits)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()


## 13. Compare held-out test metrics, per-genre scores, and confusion matrices

Primary metrics are calculated from one prediction per full track, after averaging the six window probabilities. The comparison includes track-level accuracy, balanced accuracy, macro-F1, and per-genre precision, recall, and F1. Window-level accuracy and macro-F1 are reported as secondary diagnostics, matching the baseline. Test supports can differ because each condition is restricted to its own candidate dataset.

In [ ]:
def summarize_metrics(result):
    return {
        "test_tracks": len(result["track_true"]),
        "accuracy": accuracy_score(result["track_true"], result["track_predicted"]),
        "balanced_accuracy": balanced_accuracy_score(
            result["track_true"], result["track_predicted"]
        ),
        "macro_f1": f1_score(
            result["track_true"],
            result["track_predicted"],
            average="macro",
            zero_division=0,
        ),
        "window_accuracy": accuracy_score(
            result["window_true"], result["window_predicted"]
        ),
        "window_macro_f1": f1_score(
            result["window_true"],
            result["window_predicted"],
            average="macro",
            zero_division=0,
        ),
    }


experiment_test_results = {}
experiment_reports = {}
metric_rows = {}
for run_name, run_data in EXPERIMENT_RUNS.items():
    result = collect_predictions(
        experiment_models[run_name],
        run_data["test_loader"],
        run_data["test_dataset"],
    )
    experiment_test_results[run_name] = result
    metric_rows[run_name] = summarize_metrics(result)
    experiment_reports[run_name] = classification_report(
        result["track_true"],
        result["track_predicted"],
        labels=np.arange(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

metric_comparison = pd.DataFrame(metric_rows).T
metric_comparison.index = [
    DATASET_LABELS[run_name] for run_name in metric_comparison.index
]
control_label = DATASET_LABELS["trained_on_full_dataset"]
for metric_name in ("accuracy", "balanced_accuracy", "macro_f1"):
    metric_comparison[f"{metric_name}_delta_vs_control"] = (
        metric_comparison[metric_name] - metric_comparison.loc[control_label, metric_name]
    )

print("Held-out TEST metrics (each model evaluated on its condition-specific test tracks)")
display(metric_comparison.round(3))


def make_class_metric_comparison(metric_name):
    rows = []
    for genre in CLASS_NAMES:
        full_score = experiment_reports["trained_on_full_dataset"][genre][metric_name]
        partial_score = experiment_reports[
            "trained_on_partially_filtered_dataset"
        ][genre][metric_name]
        fully_score = experiment_reports[
            "trained_on_fully_filtered_dataset"
        ][genre][metric_name]
        rows.append(
            {
                "genre": genre,
                "full_control": full_score,
                "partially_filtered": partial_score,
                "partial_delta_vs_control": partial_score - full_score,
                "fully_filtered": fully_score,
                "fully_delta_vs_control": fully_score - full_score,
                "fully_delta_vs_partial": fully_score - partial_score,
                "full_test_support": experiment_reports[
                    "trained_on_full_dataset"
                ][genre]["support"],
                "partial_test_support": experiment_reports[
                    "trained_on_partially_filtered_dataset"
                ][genre]["support"],
                "fully_test_support": experiment_reports[
                    "trained_on_fully_filtered_dataset"
                ][genre]["support"],
            }
        )
    return pd.DataFrame(rows).set_index("genre")

for metric_name in ("precision", "recall", "f1-score"):
    print(f"Per-genre {metric_name} (with deltas and test support)")
    display(make_class_metric_comparison(metric_name).round(3))

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, run_name in zip(axes, EXPERIMENT_RUNS):
    result = experiment_test_results[run_name]
    ConfusionMatrixDisplay.from_predictions(
        result["track_true"],
        result["track_predicted"],
        labels=np.arange(len(CLASS_NAMES)),
        display_labels=CLASS_NAMES,
        cmap="Blues",
        xticks_rotation=45,
        values_format="d",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(DATASET_LABELS[run_name])
plt.tight_layout()
plt.show()


## 14. Interpretation guide

Use accuracy as the primary proceed-or-discontinue signal described for this experiment. A higher filtered accuracy than the full-data control is a promising result for that condition; inspect the per-genre precision and recall tables because a decline in any genre means the approach is not necessarily better and needs further experiments. A lower accuracy means the filter did not help under this run and should be discontinued under the stated rule.

Compare the fully filtered model with the partially filtered model using accuracy, balanced accuracy, and macro-F1 together. If all three are higher, the fully filtered condition is stronger on these aggregate test metrics. If they are mixed or lower, full filtering has not shown a consistent benefit over partial filtering; examine per-genre changes and repeat the comparison.

These are separate held-out samples and a single seed. The result is exploratory, especially for genres with small test support; multiple seeds or grouped folds would be needed to establish a stable effect.

In [ ]:
partial_name = "trained_on_partially_filtered_dataset"
fully_name = "trained_on_fully_filtered_dataset"

print("Decision summary using the stated accuracy rule")
for run_name in (partial_name, fully_name):
    label = DATASET_LABELS[run_name]
    accuracy_change = metric_comparison.loc[label, "accuracy_delta_vs_control"]
    if accuracy_change > 0:
        print(
            f"- {label}: test accuracy is higher than the full-data control "
            f"(change {accuracy_change:+.3f}). Treat as promising, then review "
            "per-genre precision and recall."
        )
    elif accuracy_change < 0:
        print(
            f"- {label}: test accuracy is lower than the full-data control "
            f"(change {accuracy_change:+.3f}); under the stated rule, discontinue "
            "this filtering approach."
        )
    else:
        print(
            f"- {label}: test accuracy ties the full-data control; it shows no "
            "accuracy improvement in this run."
        )

partial_label = DATASET_LABELS[partial_name]
fully_label = DATASET_LABELS[fully_name]
aggregate_changes = {
    metric_name: (
        metric_comparison.loc[fully_label, metric_name]
        - metric_comparison.loc[partial_label, metric_name]
    )
    for metric_name in ("accuracy", "balanced_accuracy", "macro_f1")
}
print("\nFully filtered compared with partially filtered")
print(
    "Changes (fully filtered minus partial): "
    + ", ".join(f"{name} {value:+.3f}" for name, value in aggregate_changes.items())
)
if all(value > 0 for value in aggregate_changes.values()):
    print("All three aggregate metrics are higher for the fully filtered condition.")
elif all(value <= 0 for value in aggregate_changes.values()):
    print(
        "The fully filtered condition does not improve any of the three aggregate "
        "metrics in this run."
    )
else:
    print(
        "Aggregate metrics are mixed; the fully filtered condition is not consistently "
        "superior on this run."
    )

print(
    "\nInterpret these differences with the per-genre tables and test supports: "
    "each model used a different held-out sample because each split was restricted "
    "to its own dataset."
)
